# ---------- Network building notebook ----------

The goal of this notebook is to build a weighted, directed graph of the London bike-sharing network.

Author: Artur Werys

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

import networkx as nx

In [2]:
def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    for path in [current, *current.parents]:
        if (path / "Data").exists():
            return path
    raise FileNotFoundError("Could not find project root containing Data/")


PROJECT_ROOT = find_project_root()
BASE_DIR = PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / "Data"

DATA_FILE = DATA_DIR / "final_trip_data.parquet"

In [3]:
trip_data = pd.read_parquet(DATA_FILE)
trip_data.head()

,trip_id,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,bike_id,bike_model,total_duration,total_duration_ms,start_lat,start_lon,end_lat,end_lon
0,145207079,2024-12-14 23:59:00,300083,"Duke Street Hill, London Bridge",2024-12-14 23:59:00,300083.0,"Duke Street Hill, London Bridge",57763.0,CLASSIC,29s,29203.0,51.50630441,-0.087262995,51.50630441,-0.087262995
1,145207080,2024-12-14 23:59:00,1133,"Baylis Road, Waterloo",2024-12-15 00:26:00,200011.0,"Furze Green, Bow",62625.0,PBSC_EBIKE,27m 0s,1620164.0,51.50144456,-0.110699309,51.519265,-0.021345
2,145207081,2024-12-14 23:59:00,3447,"Gloucester Road (North), Kensington",2024-12-15 00:17:00,200181.0,"Richmond Way, Shepherd's Bush",62291.0,PBSC_EBIKE,18m 16s,1096981.0,51.49792478,-0.183834706,51.50035306,-0.217515071
3,145207082,2024-12-14 23:59:00,1112,"Nutford Place, Marylebone",2024-12-15 00:12:00,3422.0,"Charlbert Street, St. John's Wood",50575.0,CLASSIC,13m 3s,783763.0,51.5165179,-0.164393768,51.53430039,-0.1680743
4,145207083,2024-12-14 23:59:00,1122,"Ashley Place, Victoria",2024-12-15 00:04:00,200048.0,"Page Street, Westminster",52143.0,CLASSIC,5m 22s,322155.0,51.49616092,-0.140947636,51.493978,-0.127554


## ---------- Stations as network nodes ----------

In [4]:
start_stations_df = trip_data[
    [
        "start_station_id",
        "start_station_name",
        "start_lat",
        "start_lon",
    ]
].drop_duplicates()

start_stations_df = start_stations_df.rename(columns={
    "start_station_id": "station_id",
    "start_station_name": "station_name",
    "start_lat": "lat",
    "start_lon": "lon"
})

start_stations_df.head()

,station_id,station_name,lat,lon
0,300083,"Duke Street Hill, London Bridge",51.50630441,-0.087262995
1,1133,"Baylis Road, Waterloo",51.50144456,-0.110699309
2,3447,"Gloucester Road (North), Kensington",51.49792478,-0.183834706
3,1112,"Nutford Place, Marylebone",51.5165179,-0.164393768
4,1122,"Ashley Place, Victoria",51.49616092,-0.140947636


In [5]:
end_stations_df = trip_data[
    [
        "end_station_id",
        "end_station_name",
        "end_lat",
        "end_lon",
    ]
].drop_duplicates()

end_stations_df = end_stations_df.rename(columns={
    "end_station_id": "station_id",
    "end_station_name": "station_name",
    "end_lat": "lat",
    "end_lon": "lon"
})

end_stations_df.head()

,station_id,station_name,lat,lon
0,300083.0,"Duke Street Hill, London Bridge",51.50630441,-0.087262995
1,200011.0,"Furze Green, Bow",51.519265,-0.021345
2,200181.0,"Richmond Way, Shepherd's Bush",51.50035306,-0.217515071
3,3422.0,"Charlbert Street, St. John's Wood",51.53430039,-0.1680743
4,200048.0,"Page Street, Westminster",51.493978,-0.127554


In [6]:
stations_df = pd.concat([start_stations_df, end_stations_df], ignore_index=True).drop_duplicates()
print("Number of stations:", len(stations_df))

Number of stations: 801


## ---------- Weighted directed edges ----------

In [7]:
edges_df = trip_data.groupby(
    [
        "start_station_id",
        "start_station_name",
        "end_station_id",
        "end_station_name"
    ]
).agg(
    weight=("start_station_id", "count")
).reset_index()

In [8]:
edges_df.head(-10)

,start_station_id,start_station_name,end_station_id,end_station_name,weight
0,959,"Milroy Walk, South Bank",959.0,"Milroy Walk, South Bank",296
1,959,"Milroy Walk, South Bank",960.0,"Hop Exchange, The Borough",622
2,959,"Milroy Walk, South Bank",961.0,"Union Street, The Borough",159
3,959,"Milroy Walk, South Bank",962.0,"Stamford Street, South Bank",221
4,959,"Milroy Walk, South Bank",963.0,"Bankside Mix, Bankside",231
...,...,...,...,...,...
499733,300253,"Bermondsey Station, Bermondsey",300236.0,"Bevington Road West, North Kensington",1
499734,300253,"Bermondsey Station, Bermondsey",300237.0,"Tate Modern, Bankside",42
499735,300253,"Bermondsey Station, Bermondsey",300238.0,"Southwark Street, Bankside",8
499736,300253,"Bermondsey Station, Bermondsey",300239.0,"Victory Place, Walworth",22


## ---------- Building directed weighted graph ----------

In [9]:
graph = nx.from_pandas_edgelist(
    edges_df,
    source="start_station_name",
    target="end_station_name",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

## ---------- Creating station positions dictionary ----------


In [10]:
positions = {}

stations_df["lat"] = stations_df["lat"].astype(float)
stations_df["lon"] = stations_df["lon"].astype(float)

for _, row in stations_df.iterrows():

    positions[row["station_name"]] = (
        row["lon"],
        row["lat"]
    )